# Steane's 7-qubit quantum error correction code

Author: Adhyantha Chandrasekaran

Course: MATH 489 Intro to quantum algorithms

Instructor: Gregory Berkolaiko

## Motivation behind QEC

The primary motivation to resolve methods to correct errors in a quantum computer arises from the fact that individual physical qubits can be in the form of atoms, photons, and trapped ions. These systems are highly susceptible to get disrupted by any external factors outside the control of the computer. The best way would theoretically be to measure the qubit to monitor its state but this would collapse any entanglement or superposition essential for the quantum computer to work. Thus, we need error correcting codes that can act within the same circuit to ensure any errors from external noise can be corrected before it alters the working of the QC.

## Steane's code

Steane's code for 7 qubits is a type of CSS (Calderbank-Shor-Steane) code that is more or less an extension of the 3 qubit bit-flip (X error) error correcting code we learned in the lectures. We account for Z error (and also Y since $iXZ = Y$). We also extend this error for any arbitrary/random unitary gate for a random data line. This includes the process of generating an initial state, encoding it using the Hamming-code structure behind Steane's code, syndrome detection, correction, and decoding. The goal is to transfer the error to the "ancilla qubits", which are like temporary memory stores but need to be reversible for quantum measurements.

I am also attempting to build a GUI which allows the user to choose an initial state, apply any unitary error on one of the 7 data lines, and generate a complete error correcting circuit within a few seconds!

Without further ado, let's get right into setting up the initial state!

In [ ]:
# ----- Installations ----- #

!pip install qiskit
!pip install qiskit_aer
!pip install pylatexenc

In [ ]:
# ----- Imports ----- #

import numpy as np
import matplotlib
matplotlib.use("Agg")
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, random_unitary
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile


The total number of error bits that can be corrected is controlled by the distance term and is equal to $$t = \left\lfloor \frac{d-1}{2} \right\rfloor$$ so for $d=3$, we get $t=1$ which is as intended since we are inducing error only in one of the data lines.

Moreover, in the code cell below, the reason we use three bits is to ensure we can denote atleast 7 unique IDs to point to each one of the qubits. $2^3=8$ and the $00$ state means no error, so as expected we have exactly 7 more unique IDs that can map to a qubit upon syndrome measurement.

In [ ]:
# ----- Parameters for Steane's code ----- #

# Each of the below stabiliser generator is a list of qubit indices it acts on.
# We use [7, 1, 3] which means 7 physical qubits, 1 logical qubit, and 3 is the distance, which implies QEC on 1 qubit
# Using binary system, we can map each qubit to its corresponding binary column as written below:
#   qubit 0 -> column 001 - decimal 1
#   qubit 1 -> column 010 - decimal 2
#   qubit 2 -> column 011 - decimal 3
#   qubit 3 -> column 100 - decimal 4
#   qubit 4 -> column 101 - decimal 5
#   qubit 5 -> column 110 - decimal 6
#   qubit 6 -> column 111 - decimal 7

# Essentially, binary ID is the qubit's index + 1. This is done to satisfy the stabilizer rules as stated below:

# Stabilizer S_k acts on qubits whose column has bit k so:
#   S0 (bit 0): qubits 0,2,4,6  (columns 001,011,101,111) right has 1
#   S1 (bit 1): qubits 1,2,5,6  (columns 010,011,110,111) middle has 1
#   S2 (bit 2): qubits 3,4,5,6  (columns 100,101,110,111) left has 1
# SOURCE: https://quantum.cloud.ibm.com/learning/en/courses/foundations-of-quantum-error-correction/stabilizer-formalism/introduction

stabilizer_qubits=[
    [0, 2, 4, 6], #S0
    [1, 2, 5, 6], #S1
    [3, 4, 5, 6],#S2
]

# how to find syndrome bit: each of the three stabilizer rules output a three bit binary number. Conider qubit 5: 5 is not in S0 so that is 0. 5 is in S1 so that's 1. and 5 is in S2 so that's 1.
# We then have 110 in the order of S2, S1, S0 --> 6 in decimal. But python array starts at index 0 so 6-1 = 5 which is exactly the same qubit. Thus, the integer 5 here is called the syndrome.
# If syndrome = 0, it outputs 000 and that means no error!

syndrome_table = {s: s - 1 for s in range(1, 8)} # generated 1 through 7 and maps syndrome to qubits 0 to 6 respectively



## Building the encoder

Now, we build the framework for the encoding part of the circuit. The initial state is first created on data[0], then the encoder spreads that one qubit into the full 7-qubit logical state. This is why |1> is made with an X on data[0] before encoding, and |+> is made with a Hadamard on data[0] before encoding.

Transversal gates are still one of the nice properties of Steane's code, but for this demo we prepare the input first and then encode it so that the decoder can recover the same single-qubit state at the end.

Parity qubits and message qubits: parity bits are defined as those that have only one 1 in their binary expression since they are powers of 2, as defined by the Hamming code. So 0,1,3 are parity bits and 2,4,5,6 are message bits. Next, we swap data0 and data2 since the initial state must be encoded into the next immediately available message bit. We apply hadamards to each parity bit to put them into superposition and essentially create a complex structure that holds 2^3 parallel universes for 8 total superpositions so that the logical state can handle a single error without being destroyed.

In [ ]:
def encoder(qc, data): # returs nothing since all we do is slap on unitary and CNOT gates
  qc.swap(data[0],data[2]) # switch logical input from parity to next available message bit, from Hamming code. need to use message as a control for CNOTs so move it away from parity bit

  qc.h(data[0]) # applying superposition on three parity bits
  qc.h(data[1])
  qc.h(data[3])

  for q in (0, 1, 3, 4, 5):
      qc.cx(data[2], data[q])

  # refer to previous code cell for data bits of each parity bit
  # CNOT gates put the parity bits in entanglement with the other data bits since the parities are placed in superposition thru hadamard
  qc.cx(data[0],data[2]) #eg: 0 is control and 2 is target
  qc.cx(data[0],data[4])
  qc.cx(data[0],data[6])

  qc.cx(data[1],data[2])
  qc.cx(data[1],data[5])
  qc.cx(data[1],data[6])

  qc.cx(data[3],data[4])
  qc.cx(data[3],data[5])
  qc.cx(data[3],data[6])


In [ ]:
def decoder(qc,data): # exact opposite of encoding, to make sure we reverse all the operations performed before measurement
    qc.cx(data[3], data[6])
    qc.cx(data[3], data[5])
    qc.cx(data[3], data[4])

    qc.cx(data[1], data[6])
    qc.cx(data[1], data[5])
    qc.cx(data[1], data[2])

    qc.cx(data[0], data[6])
    qc.cx(data[0], data[4])
    qc.cx(data[0], data[2])

    for q in (5, 4, 3, 1, 0):
        qc.cx(data[2], data[q])

    qc.h(data[3])
    qc.h(data[1])
    qc.h(data[0])

    qc.swap(data[0], data[2]) # move decoded message back into data[0] for measurement

In [ ]:
# Now is the time to use the ancilla qubits previously mentioned to help extract the "syndrome" instead of directly measuring all 7 qubits, which would collapse their state.
# We initialize ancilla to |0> and bounce CNOT gates off the data lines onto the ancilla and measure the ancilla to generate a 3 bit binary integer that will map to a particular qubit
# thus telling us which qubit has the syndrome and needs to be fixed

def get_syndrome(qc,data,ancilla,xsyn,zsyn):
  # This part detects X errors (bit flips) by measuring Z-parity
  for bob, qubits in enumerate(stabilizer_qubits):
    qc.reset(ancilla[0])
    for steane in qubits:
      qc.cx(data[steane],ancilla[0])
    qc.measure(ancilla[0], xsyn[bob])

    # This part detects Z errors (phase flips) by measuring X-parity
    qc.reset(ancilla[1])
    qc.h(ancilla[1]) # Place ancilla in |+> state
    for shor in qubits:
      qc.cx(ancilla[1], data[shor])
    qc.h(ancilla[1]) # Return ancilla to Z-basis for measurement
    qc.measure(ancilla[1],zsyn[bob])

## Main simulation code

Now that we've completed the setup for the encoding, decoding, and syndrome extraction part of the circuit, we can setup the main part of the simulation which is to induce error.

In [ ]:
def decode_syndrome(syndrome):
    # A simple lookup: If 0, no error. Otherwise subtract 1 to get the qubit index.
    if syndrome == 0: # no error
        return -1
    return syndrome - 1 # find error qubit

def run_case(initial_state, error_type, target_qubit, apply_random_unitary=False):
    # Creating multiple registers like quantum hardware engineer :)
    data = QuantumRegister(7, "data") # 7 physical qubits as intended
    ancilla = QuantumRegister(2, "anc") # 2 ancilla bits
    xsyn = ClassicalRegister(3, "xsyn") # 3 qubit x and y syndrome registers (like 101 or 110)
    zsyn = ClassicalRegister(3, "zsyn")
    out = ClassicalRegister(1, "out") # 1 bit final output for measurement (0 or 1) we expect to see the same result as our initial state
    qc = QuantumCircuit(data, ancilla, xsyn, zsyn, out) # setting up the circuit

    if initial_state == "1": # user inputs 1
        qc.x(data[0]) # use X-flip to get 1 from 0
    elif initial_state == "+":
        qc.h(data[0]) # use hadamard to place in superposition

    encoder(qc, data) # encode to logical qubit after setting the initial state

    # Error injection (think parallel circuit, if one line is destroyed from error, others still survive)
    if apply_random_unitary: # like a partial rotation instead of full flipping
        U = random_unitary(2).to_instruction() # 2x2 random U converted to a gate
        qc.append(U, [data[target_qubit]]) # append unitary chunk to circuit
    else:
        if error_type == "X":
            qc.x(data[target_qubit])
        elif error_type == "Z":
            qc.z(data[target_qubit])
        elif error_type == "Y":
            qc.y(data[target_qubit]) # Reverting to direct Y gate

    # extracting the symptoms!
    get_syndrome(qc, data, ancilla, xsyn, zsyn)
    # Correcting error (FINALLY!) totally writing 14 conditional rules (7 for X and 7 for Z), also known as classical feedforward
    for syn_int in range(1, 8): # syndrome output IS an integer as established
        err_qubit = decode_syndrome(syn_int)

        with qc.if_test((xsyn, syn_int)): # Hey hardware, check if the integer on the classical register xsyn is exactly syn_int!
            qc.x(data[err_qubit])

        # If the Z-syndrome matches this value, apply Z correction
        with qc.if_test((zsyn, syn_int)):
            qc.z(data[err_qubit])

        # you might wonder why we have no error correction code for random U, but in the syndrome extraction phase, we measured the state, forcing it to collapse to no error, X, Y, or Z.
    # decode
    decoder(qc, data)

    # measure final state
    if initial_state == "+":
        qc.h(data[0])  # Apply H to measure in the X-basis

    qc.measure(data[0], out[0])
    # simulations yay!
    sim = AerSimulator()
    qc_transpiled = transpile(qc, sim)
    result = sim.run(qc_transpiled, shots=1000).result() # like flipping a coin 1000 times
    counts = result.get_counts()

    # print the results, qiskit mashes xsyn, zsyn, out together so we need to extract string
    most_frequent_str = max(counts, key=counts.get) # e.g., '0 000 000'
    parts = most_frequent_str.split() # e.g., ['0', '000', '000']

    x_syndrome_val = int(parts[2], 2)
    z_syndrome_val = int(parts[1], 2)

    total_shots = sum(counts.values())
    prob_0 = sum(v for k, v in counts.items() if k.split()[0] == "0") / total_shots
    prob_1 = sum(v for k, v in counts.items() if k.split()[0] == "1") / total_shots
    syndrome_distribution = {}

    for bit_string, shots in counts.items():
        bit_parts = bit_string.split()
        x_val = int(bit_parts[2], 2)
        z_val = int(bit_parts[1], 2)
        syndrome_distribution[(x_val, z_val)] = syndrome_distribution.get((x_val, z_val), 0) + shots

    if initial_state == "+": # check if we passed or failed, built an autograder sort of situation
        correct = (abs(prob_0 - 1.0) < 1e-6)
    elif initial_state == "1":
        correct = (abs(prob_1 - 1.0) < 1e-6)
    else:
        correct = (abs(prob_0 - 1.0) < 1e-6)

    print(f"  Initial State: |{initial_state}>")
    print(f"  Injected Error: {'Continuous Random Unitary' if apply_random_unitary else error_type} on data[{target_qubit}]")
    print(f"  Measured X-syndrome (finds X-err): {x_syndrome_val:03b}  -> corrected qubit {decode_syndrome(x_syndrome_val)}")
    print(f"  Measured Z-syndrome (finds Z-err): {z_syndrome_val:03b}  -> corrected qubit {decode_syndrome(z_syndrome_val)}")
    if apply_random_unitary:
        print("  Random U syndrome branches:")
        for (x_val, z_val), shots in sorted(syndrome_distribution.items(), key=lambda item: item[1], reverse=True):
            print(f"    X={x_val:03b}, Z={z_val:03b}: {shots / total_shots:.3f}")
    print(f"  data[0] probabilities -> |0>: {prob_0:.3f}  |1>: {prob_1:.3f}")
    print(f"  Status: {'PASSED' if correct else 'FAILED'}")

    return qc

## Testing different errors and correcting them

Now that we've built the complete circuit and logic conditions for fixing different types of errors, we can test it out to see if it works. I will be performing 5 different test cases:

1. No error
2. X error
3. Y error
4. Z error
5. Random U error

In [ ]:
print("Demo 0: No error")
# Initial state 0. We inject NO errors to prove the circuit is stable.
circuit = run_case(initial_state="0", error_type="None", target_qubit=2)
circuit.draw(output='mpl', scale=0.5, fold=-1) # just resizing to make sure it fits within google colab


Demo 0: No error
  Initial State: |0>
  Injected Error: None on data[2]
  Measured X-syndrome (finds X-err): 000  -> corrected qubit -1
  Measured Z-syndrome (finds Z-err): 000  -> corrected qubit -1
  data[0] probabilities -> |0>: 1.000  |1>: 0.000
  Status: PASSED


In [ ]:
print("Demo 1: X error (bit flip)")
circuit = run_case(initial_state="0", error_type="X", target_qubit=5)
circuit.draw(output='mpl', scale=0.7, fold=-1)


Demo 1: X error (bit flip)
  Initial State: |0>
  Injected Error: X on data[5]
  Measured X-syndrome (finds X-err): 110  -> corrected qubit 5
  Measured Z-syndrome (finds Z-err): 000  -> corrected qubit -1
  data[0] probabilities -> |0>: 1.000  |1>: 0.000
  Status: PASSED


In [ ]:
print("Demo 2: Z error (phase flip)")
circuit = run_case(initial_state="+", error_type="Z", target_qubit=6)
circuit.draw(output='mpl', scale=0.7, fold=-1)


Demo 2: Z error (phase flip)
  Initial State: |+>
  Injected Error: Z on data[6]
  Measured X-syndrome (finds X-err): 000  -> corrected qubit -1
  Measured Z-syndrome (finds Z-err): 111  -> corrected qubit 6
  data[0] probabilities -> |0>: 1.000  |1>: 0.000
  Status: PASSED


In [ ]:
print("Demo 3: Y error on superposition")
# Initial state |+>. We inject a Y error (both X and Z) on Qubit 2.
# Notice how both syndromes light up to fix it!
circuit = run_case(initial_state="+", error_type="Y", target_qubit=2)
circuit.draw(output='mpl', scale=0.7, fold=-1)


Demo 3: Y error on superposition
  Initial State: |+>
  Injected Error: Y on data[2]
  Measured X-syndrome (finds X-err): 011  -> corrected qubit 2
  Measured Z-syndrome (finds Z-err): 011  -> corrected qubit 2
  data[0] probabilities -> |0>: 1.000  |1>: 0.000
  Status: PASSED


In [ ]:
print("Demo 4: Random unitary noise")
# We inject a messy, continuous 2x2 unitary matrix of errors on Qubit 1.
# Watch the measurement force it to collapse and fix it anyway!
circuit = run_case(initial_state="1", error_type="None", target_qubit=1, apply_random_unitary=True)
circuit.draw(output='mpl', scale=0.7, fold=-1) # print it all in one row and not as separate chunks


Demo 4: Random unitary noise
  Initial State: |1>
  Injected Error: Continuous Random Unitary on data[1]
  Measured X-syndrome (finds X-err): 000  -> corrected qubit -1
  Measured Z-syndrome (finds Z-err): 000  -> corrected qubit -1
  Random U syndrome branches:
    X=000, Z=000: 0.403
    X=010, Z=000: 0.277
    X=000, Z=010: 0.176
    X=010, Z=010: 0.144
  data[0] probabilities -> |0>: 0.000  |1>: 1.000
  Status: PASSED
